# Closed Sports-market calibration

Uses the whole closed Sports catalogue—no title filter or sampling. Source data are in `closed_sports_all_data.csv`: one row per market, with separate UTC time bounds and activity metrics for each of the 24 hours before close. This notebook derives one calibration forecast per usable settled binary market from the nearest hour with a recorded trade, then saves a compact reusable forecast CSV. The master CSV is read in chunks and only the required columns are loaded. Volume and liquidity are final catalogue snapshots; hourly trade notional is a trading-activity proxy, not historical market volume. Sport groups are inferred from title keywords.

In [ ]:
from pathlib import Path
import subprocess, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'prepare_sports_forecasts.py').exists()), None)
if ROOT is None:
    raise FileNotFoundError('Open this notebook from the polymarket_school_project folder.')
MASTER = ROOT / 'data/all_closed_sports/closed_sports_all_data.csv'
CACHE = ROOT / 'data/all_closed_sports/closed_sports_market_forecasts.csv'
if not MASTER.exists():
    raise FileNotFoundError(f'Master dataset not found: {MASTER}')
if not CACHE.exists() or MASTER.stat().st_mtime > CACHE.stat().st_mtime:
    subprocess.run([sys.executable, str(ROOT / 'prepare_sports_forecasts.py'), '--force'], check=True)

data = pd.read_csv(CACHE)
data['market_id'] = data['market_id'].astype(str)
print(f"Catalogue liquidity snapshot present for {data['liquidityNum'].notna().sum():,} of {len(data):,} forecasts")

data['bin'] = np.minimum((data['probability'].clip(0, 1) * 20).astype(int), 19)
bins = data.groupby('bin').agg(n=('actual', 'size'), mean_probability=('probability', 'mean'), observed_rate=('actual', 'mean'))
n, rate, z = bins['n'].to_numpy(), bins['observed_rate'].to_numpy(), 1.96
denominator = 1 + z**2 / n
center = (rate + z**2 / (2*n)) / denominator
margin = z * np.sqrt(rate*(1-rate)/n + z**2/(4*n**2)) / denominator
bins['ci_low'], bins['ci_high'] = center - margin, center + margin
X = np.column_stack([np.ones(len(bins)), bins['mean_probability'].to_numpy()])
weights = np.sqrt(bins['n'].to_numpy())
intercept, slope = np.linalg.lstsq(X * weights[:, None], bins['observed_rate'].to_numpy() * weights, rcond=None)[0]

fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
grid = np.linspace(0, 1, 101)
ax.plot([0, 1], [0, 1], '--', color='0.45', label='Perfect calibration')
ax.plot(grid, intercept + slope * grid, color='#c06b32', label=f'Weighted regression; slope={slope:.2f}')
overall_yerr = np.maximum(np.vstack([bins.observed_rate-bins.ci_low, bins.ci_high-bins.observed_rate]), 0)
ax.errorbar(bins.mean_probability, bins.observed_rate, yerr=overall_yerr, fmt='none', ecolor='#3569a8', elinewidth=1.5, capsize=5, label='Wilson 95% CI', zorder=3)
ax.scatter(bins.mean_probability, bins.observed_rate, s=22, color='#3569a8', edgecolor='white', linewidth=.6, label='5% bin mean', zorder=4)
for bin_id, row in bins.iterrows():
    ax.annotate(f'n={row.n:,}', (row.mean_probability, row.observed_rate), xytext=(4, 4), textcoords='offset points', fontsize=8)
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel='Predicted probability', ylabel='Observed event frequency', title=f'Closed Sports markets: calibration (n={len(data):,})')
ax.grid(alpha=.2)
ax.legend()
fig.savefig(ROOT / 'figures/closed_sports_calibration.png', dpi=180)
plt.show()

table = bins.copy()
table.index = [f'{5*i}–{5*(i+1)}%' for i in table.index]
table.index.name = 'Predicted probability'
print(f'Loaded {len(data):,} forecasts from {CACHE.name}; regression: observed = {intercept:.3f} + {slope:.3f} × predicted')
display(table.round(3))

# Sport is inferred from question and market slug; ambiguous titles remain unclassified.
sport_rules = [
    ('American football', r'\b(?:NFL|CFB|NCAA Football|Super Bowl|college football|pro football draft|quarterback|starting QB|NFL Draft|touchdown|highest scoring game of Week \d+|fantasy points in Week \d+|combined points scored|Army and Air Force combine|USC and Washington combine|Chiefs|Eagles|Cowboys|Giants|Rams|Seahawks|Vikings|Buccaneers|Broncos|Texans|Colts|Patriots|Chargers|Saints|Ravens|Commanders|Raiders|Jaguars|Bears|Packers|49ers|Bills|Dolphins|Jets|Bengals|Browns|Steelers|Titans|Falcons|Panthers|Cardinals|Lions|Penn State|SMU|Baylor|BYU|Kansas|Alabama|Missouri|Arkansas|Mississippi State|Army|Air Force|Washington|Texas|Georgia|Florida|UCF|Notre Dame|Ohio State|Oklahoma State|West Virginia|Arizona|Arizona State|Texas Tech|Kentucky|Iowa State|Nebraska|Illinois|St\. John\'s)\b'),
    ('Basketball', r'\b(?:NBA|WNBA|basketball|EuroLeague|NCAA Basketball|NCAA Tournament|March Madness|Sweet 16|Final Four|conference tournament|UConn|Connecticut|NBA Draft|NCAAB|Timberwolves|Thunder|Knicks|Celtics|Cavaliers|Magic|Lakers|Clippers|Warriors|Suns|Mavericks|Nuggets|Bucks|Bulls|Heat|76ers|Sixers|Hawks|Hornets|Nets|Pistons|Pacers|Wizards|Raptors|Grizzlies|Pelicans|Rockets|Spurs|Trail Blazers|Blazers|Kings|Jazz)\b'),
    ('Basketball', r'\b(?:vs\. .*combine for 1\d{2}\+? points|Houston vs\. Duke)\b'),
    ('Basketball', r'\bDonovan Mitchell\b'),
    ('Basketball', r'\b(?:NIT Championship|Chattanooga vs\. UC Irvine|Iowa advance to the Quarterfinals)\b'),
    ('Basketball', r'\b(?:ACC title|Jayson Tatum|NCAA Championship game in scoring|Duke win the 2026 ACC)\b'),
    ('Basketball', r'\b(?:LaMelo Ball|LeBron James|Celebrity All-Star Game MVP)\b'),
    ('Basketball', r'\b(?:Kevin Durant|Kyrie Irving|Miami and Duke combine)\b'),
    ('Basketball', r'\b(?:Karl-Anthony Towns|Villanova)\b'),
    ('Basketball', r'\b(?:Naismith Men\'s College Player of the Year|Cameron Boozer)\b'),
    ('Basketball', r'\b(?:SGA attempt|Shai Gilgeous-Alexander|free throws in the game)\b'),
    ('American football', r'\b(?:Fantasy Football|rush for \d[,.]?\d* yards|Saquon Barkley|Tyreek Hill|Travis Kelce|J\.J\. McCarthy|Jalen Hurts|Dak Prescott)\b'),
    ('American football', r'\b(?:passing yards|players drafted|top.*players drafted|Scorigami|Big Game - Winning Conference|Ohio St\.|LSU|Oregon|Hawaii|Nevada)\b'),
    ('American football', r'\b(?:Heisman Trophy|Arch Manning|Jaxson Dart|Shedeur Sanders|Big Game)\b|pro-football-championship'),
    ('American football', r'\b(?:Fantasy Wide Receiver|Garrett Wilson|Justin Jefferson|DK Metcalf|D\.K\. Metcalf)\b'),
    ('American football', r'\b(?:play in Week 1|Chris Olave|Shilo Sanders)\b'),
    ('American football', r'\b(?:Aaron Rodgers|Christian McCaffrey|play Week \d|play again by December)\b'),
    ('American football', r'\bTravis Etienne\b'),
    ('American football', r'\b(?:first TD|Ole Miss|Jonathan Gannon|Odell Beckham Jr\.|Quinnen Williams|Breece Hall|Patrick Mahomes)\b'),
    ('American football', r'\b(?:score a TD|Amon Ra St Brown|Amon-Ra St\. Brown)\b'),
    ('American football', r'\b(?:Clemson|Wake Forest)\b'),
    ('Baseball', r'\b(?:MLB|baseball|World Series|Rockies|Rays|Yankees|Dodgers|Red Sox|Cubs|Mets|Braves|Phillies|Guardians|Royals|Pirates|Cardinals|Tigers|Astros|Mariners|Orioles|Twins|Reds|Padres|Giants|Athletics|Blue Jays|White Sox|Diamondbacks|Nationals|Marlins|Rangers|next team be)\b'),
    ('Baseball', r'\brun scored in the first inning\b'),
    ('Hockey', r'\b(?:NHL|hockey|Stanley Cup|KHL|AHL|IIHF)\b'),
    ('Hockey', r'\b(?:Matthew Tkachuk|score 1\+ goals against the Oilers|Edmonton Oilers)\b'),
    ('Hockey', r'\b(?:Memorial Cup|London Knights|Moncton Wildcats)\b'),
    ('Tennis', r'\b(?:tennis|ATP|WTA|Wimbledon|Grand Slam|ITF|French Open|Australian Open|Indian Wells Open|Miami Open|Madrid Open|Davis Cup|Mallorca Championships|Veolia Arizona Open|Draper vs\. Machac|mixed doubles|singles|Swiss Open: .* vs\.|US Open: .* vs\.|Open: .* vs\.)'),
    ('Tennis', r'\b(?:Hangzhou Open|Casper Ruud win the 2025 Madrid Open|Novak Djokovic win the 2025 Miami Open)\b'),
    ('Tennis', r'\b(?:Terra Wortmann Open|Daniil Medvedev)\b'),
    ('Tennis', r'\b(?:Alcaraz win 3:[01]|Taylor Fritz|Jannik Sinner)\b'),
    ('Tennis', r'\bUS Open: .* and .*\b'),
    ('Tennis', r'\b(?:HSBC Championships|Boss Open|Alexander Zverev|Jiri Lehecka)\b'),
    ('Tennis', r'\b(?:Alexander Bublik|Swiss Open)\b'),
    ('Combat sports', r'\b(?:UFC|MMA|boxing|wrestling|Power Slap|WrestleFest|fight end in (?:Round|the Round)|win by KO|knockdowns? during|KO/TKO|vs\. .*\b(?:fight|round|KO|decision))\b'),
    ('Combat sports', r'\b(?:Pavlovich vs\.|Rozenstruik|Paige VanZant|Covington vs\.|Buckley|Nurmagomedov vs\.|Garcia vs\. Haney|Teofimo Lopez|O\'Malley vs\.|Mairon Santos)\b'),
    ('Combat sports', r'\b(?:Pennington vs\.|Peña|Allen vs\. Hernandez|Stoltzfus vs\.|Ruziboev|Anthony Joshua win in Round|Anthony Joshua|Tyson Fury|Topuria|Holloway)\b'),
    ('Combat sports', r'\b(?:Santos vs\. Tate|Lemos vs\.|Lucindo|Green vs\. Ruffy|Pantoja vs\. Asakura|Ziam vs\. Davis|Ryan Garcia vs\. Rolly Romero)\b'),
    ('Combat sports', r'\b(?:Nathan Fletcher vs\.|Caolán Loughran|Silva vs\. Almeida|Hill vs\. Ricci|Emmers vs\. Miranda|win in Round \d)\b'),
    ('Combat sports', r'\b(?:Murzakanov vs\. Ribeiro|Ankalaev vs\. Rakic|Nikita Krylov vs\. Dominick Reyes|Vergil Ortiz Jr\. vs\. Israil Madrimov|Naimov vs\. Ofli|Brunno Ferreira vs\. Armen Petrosyan|Ponzinibbio vs\. Harris|Yair Rodriguez vs\. Patricio Pitbull)\b'),
    ('Combat sports', r'\b(?:fight reach the \d+(?:st|nd|rd|th) round|fight Jake Paul|Silva vs\. Van|Rosas Jr\. vs\. Morales)\b'),
    ('Combat sports', r'\b(?:Rodriguez vs\. Costa|Bonfim vs\. Williams|Christian Leroy Duncan vs\. Andrey Pulyaev|Alex Morono vs\. Carlos Leal|Jake Paul knockout Mike Tyson|knockout Mike Tyson)\b'),
    ('Combat sports', r'\b(?:Medina vs\. Gautier|Emmett vs\. Murphy)\b'),
    ('Combat sports', r'\b(?:Harrison vs\. Vieira|Johnson vs\. Azaitar)\b'),
    ('Combat sports', r'\b(?:Kape vs\. Almabayev|Santos vs\. Lee|Du Plessis vs\. Adesanya|Oezdemir vs\. Ulberg|Kape vs\. Silva|Moreno vs\. Albazi)\b'),
    ('Cricket', r'\b(?:cricket|IPL|International Cricket|Delhi Capitals|Sunrisers Hyderabad)\b'),
    ('Golf', r'\b(?:golf|PGA|LPGA|Masters Tournament|Masters winner.*amateur|highest finishing debutant.*Masters|first round leader|finish in the Top \d+|leader after Round \d|lowest round|shoot the lowest round|shoot the best round|finish as leader|DP World Tour|LIV Golf|Arnold Palmer Invitational|FedEx St\. Jude|Travelers Championship|Rocket Classic|Cognizant Classic|RBC Heritage|RBC Canadian Open|THE CJ CUP Byron Nelson|Tour Championship|Pinnacle Bank Championship|Danish Golf Championship|Mauritius Open|Ryder Cup|DP AfrAsia|John Deere Classic|Valspar Championship|Charles Schwab Challenge|WM Phoenix Open|The Players Championship|Puerto Rico Open|Corales Puntacana|Valero Texas Open|BMW Championship|Memorial Tournament|ISCO Championship|Texas Children\'s Houston Open|Myrtle Beach Classic|Cadillac Championship|Genesis Scottish Open|Genesis Invitational|Cognizant Classic|Charles Schwab|WM Phoenix|Houston Open|Wyndham Championship|Sony Open|AT\&T Pebble Beach Pro-Am|Biltmore Championship|The American Express tournament|The Open Championship|(?:US|U\.S\.) Open.*(?:finish|Top \d+|leader|lowest round|best round|win)|(?:leader|lowest|best|round|finish).*(?:US|U\.S\.) Open)\b'),
    ('Golf', r'\b(?:RBC Canadian Open|3M Open|Truist Championship|Farmers Insurance Open|Nordea Open|THE CJ CUP Byron Nelson|(?:The )?Open Championship)\b'),
    ('Golf', r'\b(?:Hero World Challenge|The Masters|win the 20\d{2} Masters)\b'),
    ('Golf', r'\b(?:Hole in 1 Challenge|complete the Hole-in-1 Challenge)\b'),
    ('Golf', r'\b(?:hole in one|Presidents Cup|another golfer score the most points|Masters winner be a LIV player)\b'),
    ('Golf', r'\b(?:LIV Greenbrier|LIV Golf)\b'),
    ('Golf', r'\b(?:LIV Riyadh Tournament|LIV.*Tournament)\b'),
    ('Golf', r'\b(?:World Wide Technology Championship|Steven Fisk)\b'),
    ('Motorsports', r'\b(?:F1|Formula 1|NASCAR|MotoGP|IndyCar|Formula E|Grand Prix|Canadian GP|Italy GP|Indy 500|Indianapolis 500|Bommarito Automotive Group 500|Great American Getaway 400|pole|24 Hours of Le Mans|Verstappen|Leclerc|Lewis Hamilton|Bortoleto|DeFrancesco|Hauger|Castroneves|Andretti|Armstrong)\b'),
    ('Motorsports', r'\b(?:Constructors.? Championship|Hamilton vs\. Russell|Mercedes finish second in the 20\d{2})\b'),
    ('Motorsports', r'\b(?:highest Constructor score|Haas have the highest Constructor score|McLaren have the highest Constructor score)\b'),
    ('Motorsports', r'\b(?:Drivers Championship|another driver finish second)\b'),
    ('Motorsports', r'\b(?:Alonso vs\. Stroll|Fernando Alonso|Lance Stroll)\b'),
    ('Motorsports', r'\b(?:Tsunoda vs\. Hadjar|Yuki Tsunoda|Isack Hadjar)\b'),
    ('Cycling', r'\b(?:cycling|Tour de France|Tour of Flanders|Tour de Flanders|Giro d’Italia|Vuelta a España|UCI|BMX)\b'),
    ('Horse racing', r'\b(?:horse racing|Kentucky Derby|Preakness|Belmont Stakes|Queen Anne Stakes|Queen Elizabeth II Jubilee Stakes|Grand National|Jockey Club Gold Cup|Prince of Wales Stakes|WSOP main event)\b'),
    ('Horse racing', r'\bBreeders.? Cup Classic\b'),
    ('Horse racing', r'\b(?:Ascot Gold Cup|Ascot Stakes)\b'),
    ('Horse racing', r'\bTownsville Cup\b'),
    ('Swimming', r'\b(?:swimming|butterfly|freestyle|backstroke|breaststroke|medley|World Aquatics)\b'),
    ('Athletics', r'\b(?:athletics|track and field|marathon|hurdles|high jump|pole vault|long jump|triple jump|shot put|hammer throw|javelin|discus|decathlon|heptathlon|\d{2,4}m [MW])\b'),
    ('Athletics', r'\b(?:gold in \d{2,4}m)\b'),
    ('Athletics', r'\b(?:run the 5k|5k in under|5K race)\b'),
    ('Gymnastics', r'\b(?:gymnastics|vault final|uneven bars?|balance beam|floor exercise)\b'),
    ('Rugby', r'\b(?:rugby|Six Nations|Super Rugby)\b'),
    ('Handball', r'\b(?:handball|EHF)\b'),
    ('Volleyball', r'\b(?:volleyball|FIVB)\b'),
    ('Esports', r'\b(?:esports|League of Legends|Dota 2|Counter-Strike|Valorant|CS2|StarCraft|Rainbow Six|Call of Duty|Honor of Kings|Overwatch|Rocket League|Mobile Legends|MLBB|LEC|MPL|NACL|LCK|LPL|LCS|CBLOL|VCT|EWC|ESL|LOL Worlds|DreamLeague|PGL Wallachia|HLTV|BLAST|MSI|Team Liquid|Elisa Masters|BLAST\.tv|EMEA Masters|Masters Santiago|match point finals|IEM|FRAG Midwest|Blast Premier|BLAST Slam|FalleN|TYLOO|The MongolZ|G2)\b'),
    ('Esports', r'\b(?:PGL|First Stand|LCP|MOUZ|Deep Cross Gaming|Caps win MVP|Kiin win MVP|KSCERATO|Ruler win MVP)\b'),
    ('Esports', r'\b(?:GeoGuessr World Championship|duWapcs get exposed for cheating)\b'),
    ('Esports', r'\b(?:LoL:|GEN vs\. T1|Gen\.G|T1)\b'),
    ('Esports', r'\b(?:Intel Extreme Masters|Team Vitality|Valve add Cache to the Map Pool)\b'),
    ('Esports', r'\bTyler 1 All Star tournament\b'),
    ('Esports', r'\b(?:DreamHack Major|Toronto KOI)\b'),
    ('Esports', r'\b(?:CS Asia Championships|MattCS get exposed for cheating|Legacy win CS)\b'),
    ('Esports', r'\b(?:Astralis|roster change before July)\b'),
    ('Chess', r'\b(?:chess|FIDE|Title Tuesday|World Rapid Championship|World Blitz Championship|Carlsen vs\. Abdusattorov)\b'),
    ('Chess', r'\bSinquefield Cup\b'),
    ('Darts', r'\b(?:darts|PDC World Darts Championship)\b'),
    ('Rodeo', r'\b(?:rodeo|bull riding)\b'),
    ('Competitive eating', r'\b(?:Nathan\'s Famous Hot Dog Eating Contest|hot dog eating contest)\b'),
    ('Competitive eating', r'\b(?:eat the most hot dogs|eating contest)\b'),
    ('Competitive eating', r'\b(?:Joey Chestnut|Nathan\'s Famous Hot Dog 2025)\b'),
    ('Lacrosse', r'\b(?:lacrosse|PLL)\b'),
    ('Pickleball', r'\b(?:pickleball|Major League Pickleball)\b'),
    ('Pickleball', r'\b(?:PPA: Indoor National Championships|PPA Indoor|Federico Staksrud)\b'),
    ('Poker', r'\b(?:poker|WSOP)\b'),
    ('Poker', r'\b(?:Hustler Casino Live|Daniel Negreanu Profit)\b'),
    ('Poker', r'\bHustler Casino\b'),
    ('Action sports', r'\b(?:X Games|skateboard|skateboarding|BMX Freestyle)\b'),
    ('Olympics', r'\b(?:Olympics|Olympic Games|Winter Games|Paralympics|Paralympic Games)\b'),
    ('Olympics', r'\b(?:most medals|3rd most medals)\b'),
    ('Soccer', r'\b(?:soccer|FIFA|UEFA|EPL|UCL|UEL|NWSL|Premier League|La Liga|Bundesliga|Serie A|MLS|Eredivisie|Champions League|Europa League|Conference League|Copa del Rey|Carabao Cup|Club World Cup|football association|starting (?:11|XI)|Anytime Goalscorer|Both Teams to Score|draw at halftime|end in a draw|end in extra time|the match be a draw|first goal|top scorer|90 min|2024 Euros|Euros|transfer window|Manchester United|Manchester City|Real Madrid|Barcelona|Bayern Munich|Borussia Dortmund|Bayer Leverkusen|AC Milan|Inter Milan|Atalanta|Juventus|PSG|Paris Saint-Germain|Liverpool|Arsenal|Chelsea|Tottenham|Newcastle United|Aston Villa|Crystal Palace|Brentford|Wolverhampton|Brighton|Everton|Fulham|Nottingham Forest|West Ham|LOSC Lille|Lille|Feyenoord|Lazio|Girona|Ferencvaros|Fenerbahce|RB Salzburg|Red Star Belgrade|Young Boys|Dinamo Zagreb|Slovan Bratislava|PSV Eindhoven|Stade Brestois|River Plate|Fluminense|Malmo FF|Rangers|Stuttgart|Nice|Leicester City|Southampton|Bournemouth|St Louis City SC|Portland Timbers|FK RFS|FCSB|Atletico Madrid|Atlético Madrid|Real Sociedad|Shakhtar Donetsk|Bologna|Union St-Gilloise|Twente|Celtic|RB Leipzig|Spain vs\.|Portugal|Poland|France|England|Netherlands|Uruguay|Peru|Suriname|Dominican Republic|Bosnia|San Marino|Bolivia|Honduras|El Salvador|Brazil|Japan|Switzerland)\b|\bFC\b'),
    ('Soccer', r'\b(?:Bodo Glimt|Union Saint-Gilloise|Sporting Kansas City|Seattle Sounders|AZ Alkmaar|Ludogorets|Ajax|Qarabag|Club Brugge|Sporting CP|Dynamo Kyiv|Roma|Scotland|Croatia|Slovakia|Chile|Finland U21|Denmark U21|Celtic|RB Leipzig)\b'),
    ('Soccer', r'\b(?:CONCACAF Gold Cup|Inter Miami|Nashville SC|Real Salt Lake|D\.C\. United|Midtjylland|Hoffenheim|Atletico de San Luis|AS Saint-Etienne|Cristiano Ronaldo|Marcus Rashford|Vinicius Junior|Adam Wharton|Bruno Fernandes|Mexico win the 2025 CONCACAF)\b'),
    ('Soccer', r'\b(?:Viktoria Plzen|PAOK|RFS|Maccabi TLV|Slavia Praha|Athletic Bilbao|Ballon d\'Or|win \d-\d)\b'),
    ('Soccer', r'\b(?:Wales beat Iceland|Saudi Arabia beat Haiti|Vancouver Whitecaps|Columbus Crew|LAFC|Colorado Rapids|Benfica|Auckland City|Cade Cowell|score a goal in the 2026 Sidemen Charity Match)\b'),
    ('Soccer', r'\b(?:Ipswich Town and Newcastle|Ipswich Town|Belgium win|Hungary win|Haiti beat Saudi Arabia)\b'),
    ('Soccer', r'\b(?:Empoli|Fiorentina|Braga|IF Elfsborg|Lionel Messi|Lamine Yamal|Ballon d)\b'),
    ('Soccer', r'\b(?:Dortmund|Bayern|Mohamed Salah|Mohammed Salah|Mexico beat USA in Penalties)\b'),
    ('Soccer', r'\b(?:CAF Africa Cup of Nations|Sidemen Charity Match|official 2026 World Cup squad|Frankfurt beat Lyon)\b'),
    ('Soccer', r'\b(?:Euro 2024|Player of the Tournament|Argentina win the World Cup)\b'),
    ('Soccer', r'\b(?:Belgium beat Italy|Trinidad and Tobago beat USA|Guadeloupe beat Jamaica|Italy U21 beat Spain U21)\b'),
    ('Soccer', r'\b(?:Lyon beat Besiktas|Palmeiras beat Al Ahly|Romania U21 beat Spain U21|Mexico win \d[–-]\d|[A-Z][a-z]+ U21 beat [A-Z][a-z]+ U21)\b'),
    ('Soccer', r'\b(?:Lyon|Besiktas|Palmeiras|Al Ahly|Pafos|Flamengo|Esperance Sportive de Tunis|3\+ Goals Scored|World Cup Final: Weather Delay)\b'),
    ('Soccer', r'\b(?:LOSC|Marseille)\b'),
    ('Soccer', r'\b(?:World Cup: Largest Margin of Victory|Lithuania beat Malta)\b'),
    ('Australian rules football', r'\b(?:AFL|Fremantle Dockers|Port Adelaide Power)\b'),
    ('Climbing', r'\b(?:Alex Honnold|free solo Taipei 101)\b'),
    ('Soccer', r'\b(?:CONCACAF Gold Cup|Inter Miami|Nashville SC|Real Salt Lake|D\.C\. United|Midtjylland|Hoffenheim|Atletico de San Luis|AS Saint-Etienne|Cristiano Ronaldo|Marcus Rashford|Vinicius Junior|Adam Wharton|Bruno Fernandes)\b'),
]
sport_text = data['question'].fillna('').astype(str) + ' ' + data['slug'].fillna('').astype(str)
data['sport'] = 'Other / unclassified'
combined_sport_rules = {}
for label, pattern in sport_rules:
    combined_sport_rules.setdefault(label, []).append(f'(?:{pattern})')
for label, patterns in combined_sport_rules.items():
    match = sport_text.str.contains('|'.join(patterns), case=False, regex=True, na=False)
    data.loc[data['sport'].eq('Other / unclassified') & match, 'sport'] = label

# Many event slugs use short league/game codes instead of full sport names.
slug_prefix_rules = [
    ('Soccer', r'^(?:fifwc|lal|arg|sea|elc|bra|uel|fif|mex|unl|fl1|bun|es2|clf|chi|kor|bra2|ere|spl|por|nor|swe|tur|bl2|jap|itsb|sud|enl|nwsl|col|col1|fr2|argpn|ja2|rus|lib|den|j2100|uwcl|ukr1|rou1|per1|scop|el2|efl|el1|egy1|efa|conl|uslc|mar1|ned2|chi1|swe2|j1100|ecu1|cze1|gtm|pol|bol1|bel1|kor2|brco|bul|tur2|sui|nor2|uzb1|col2|hr1|ven1|uru1|aut|isl1|itc|dfb|chl2|kaz1|u20wwc|svk1|srb|saf1|irl1|bra3|isr|hun|lva1|est1|acle|usl1|gre1|ltu1|grc|idn1|wsl|uae1|slo|isp|nlc|aze1|aze2|bel2|uef|wwcquefa|qat1|cde|nirl1|skc|sclc|canpl|cdr|asean|argcopa|fro1|caf|idn2|acn|chfa|scoc|cof|cafcl|usc|gsc|usoc|frtc|ptsc|icwq|fin1|ptc|fpd|chi2|atc|ccup|ccc|ucol|auc|trsk|canch|afc|epl|ucl|tha1|aus|geo1|ire|ssc|tpe1|aseanc|concacaf|euro|copa|psg|fa|cl)-'),
    ('Soccer', r'^(?:con)-'),
    ('Esports', r'^(?:lol|dota2|cs2|val|sc2|r6siege|codmw|ow|rl|lec|hok|mlbb|blasttv|ecs|iem|pgl|msi|fissure|blast|the)-'),
    ('Basketball', r'^(?:cbb|cwbb|ncaab|ucol|march|bk[a-z0-9]+)-'),
    ('American football', r'^(?:cfb)-'),
    ('Baseball', r'^(?:npb|kbo|cpbl|lmb|alds|nlds)-'),
    ('Hockey', r'^(?:ahl|khl|shl|dehl|cehl|snhl|mwoh|wwoh|wch)-'),
    ('Golf', r'^(?:dpwt|liv|hsbc|kft)-'),
    ('Tennis', r'^(?:itf|french|australian|internazionali|terra)-'),
    ('Cricket', r'^(?:crban|crint|cricipl|cric[a-z0-9]*)-'),
    ('Combat sports', r'^(?:powerslap|wrestlefest)-'),
    ('Chess', r'^(?:fide)-'),
    ('Lacrosse', r'^(?:pll|wll)-'),
]
slug_text = data['slug'].fillna('').astype(str)
for label, pattern in slug_prefix_rules:
    match = slug_text.str.contains(pattern, case=False, regex=True, na=False)
    data.loc[data['sport'].eq('Other / unclassified') & match, 'sport'] = label
# Reproducible 1,000-market residual audit; inspect these examples before adding another rule.
other_audit_sample = data.loc[data['sport'].eq('Other / unclassified'), ['market_id', 'question', 'slug']].sample(n=min(1000, data['sport'].eq('Other / unclassified').sum()), random_state=20261028)
print(f"Residual audit: sampled {len(other_audit_sample):,} of {(data['sport'] == 'Other / unclassified').sum():,} unclassified markets; showing first 50.")
display(other_audit_sample.head(50))

# Per-sport calibration plots, using the same 5% bins, Wilson intervals, and weighted regression as the overall chart.
SPORT_CALIBRATION_DIR = ROOT / 'figures/calibration_by_sport'
SPORT_CALIBRATION_DIR.mkdir(parents=True, exist_ok=True)
sport_counts = data.groupby('sport').size().sort_values(ascending=False)
sports_to_plot = sport_counts.index[sport_counts.index != 'Other / unclassified']
print(f'Saving calibration plots for all {len(sports_to_plot)} inferred sports; unclassified markets are excluded.')

for sport in sports_to_plot:
    sport_data = data.loc[data['sport'].eq(sport)].copy()
    sport_data['bin'] = np.minimum((sport_data['probability'].clip(0, 1) * 20).astype(int), 19)
    sport_bins = sport_data.groupby('bin').agg(n=('actual', 'size'), mean_probability=('probability', 'mean'), observed_rate=('actual', 'mean'))
    sport_bins = sport_bins.loc[sport_bins['n'].gt(0)].copy()
    bn, br, bz = sport_bins['n'].to_numpy(), sport_bins['observed_rate'].to_numpy(), 1.96
    bden = 1 + bz**2 / bn
    bcenter = (br + bz**2 / (2*bn)) / bden
    bmargin = bz * np.sqrt(br*(1-br)/bn + bz**2/(4*bn**2)) / bden
    sport_bins['ci_low'], sport_bins['ci_high'] = bcenter - bmargin, bcenter + bmargin
    Xs = np.column_stack([np.ones(len(sport_bins)), sport_bins['mean_probability'].to_numpy()])
    ws = np.sqrt(sport_bins['n'].to_numpy())
    sport_intercept, sport_slope = np.linalg.lstsq(Xs * ws[:, None], sport_bins['observed_rate'].to_numpy() * ws, rcond=None)[0]

    fig, ax = plt.subplots(figsize=(7, 6), constrained_layout=True)
    grid = np.linspace(0, 1, 101)
    ax.plot([0, 1], [0, 1], '--', color='0.45', label='Perfect calibration')
    ax.plot(grid, sport_intercept + sport_slope * grid, color='#c06b32', label=f'Weighted regression; slope={sport_slope:.2f}')
    sport_yerr = np.maximum(np.vstack([sport_bins.observed_rate-sport_bins.ci_low, sport_bins.ci_high-sport_bins.observed_rate]), 0)
    ax.errorbar(sport_bins.mean_probability, sport_bins.observed_rate, yerr=sport_yerr, fmt='none', ecolor='#3569a8', elinewidth=1.5, capsize=5, label='Wilson 95% CI', zorder=3)
    ax.scatter(sport_bins.mean_probability, sport_bins.observed_rate, s=22, color='#3569a8', edgecolor='white', linewidth=.6, label='5% bin mean', zorder=4)
    for _, row in sport_bins.iterrows():
        ax.annotate(f'n={row.n:,}', (row.mean_probability, row.observed_rate), xytext=(4, 4), textcoords='offset points', fontsize=8)
    ax.set(xlim=(0, 1), ylim=(0, 1), xlabel='Predicted probability', ylabel='Observed event frequency', title=f'{sport}: calibration (n={len(sport_data):,})')
    ax.grid(alpha=.2)
    ax.legend()
    filename = ''.join(c.lower() if c.isalnum() else '_' for c in sport).strip('_')
    fig.savefig(SPORT_CALIBRATION_DIR / f'{filename}.png', dpi=180)
    plt.show()
    plt.close(fig)

display(sport_counts.rename('markets').to_frame().assign(plotted=lambda x: x.index.isin(sports_to_plot)))
data['brier'] = (data['probability'] - data['actual']) ** 2
sport_stats = data.groupby('sport').agg(n=('brier', 'size'), brier=('brier', 'mean'), sd=('brier', 'std'))
sport_stats = sport_stats.loc[sport_stats['n'] >= 500].sort_values('brier')
sport_stats['ci'] = 1.96 * sport_stats['sd'] / np.sqrt(sport_stats['n'])
fig, ax = plt.subplots(figsize=(8, max(4, .38 * len(sport_stats))), constrained_layout=True)
y = np.arange(len(sport_stats))
ax.errorbar(sport_stats['brier'], y, xerr=sport_stats['ci'], fmt='o', capsize=3, color='#3569a8')
ax.set_yticks(y, [f'{name} (n={row.n:,})' for name, row in sport_stats.iterrows()])
ax.set(xlabel='Mean Brier score (lower is better)', title='Forecast accuracy by title-inferred sport')
ax.grid(axis='x', alpha=.25)
fig.savefig(ROOT / 'figures/sports_brier_by_sport.png', dpi=180)
plt.show()

# Accuracy by market-volume/liquidity quartiles and final-hour activity quartiles.
metric_specs = [
    ('Lifetime volume snapshot', 'volumeNum'),
    ('Liquidity snapshot', 'liquidityNum'),
    ('Final-hour trade notional', 'latest_hour_trade_side_notional_usdc'),
]
fig, axes = plt.subplots(1, len(metric_specs), figsize=(13, 4.5), constrained_layout=True)
for ax, (title, column) in zip(axes, metric_specs):
    values = pd.to_numeric(data[column], errors='coerce')
    valid = values.notna() & values.ge(0)
    if values.loc[valid].nunique() < 2:
        ax.set_title(f'{title} (not enough distinct values)')
        continue
    subset = data.loc[valid, ['brier']].copy()
    subset['quartile'] = pd.qcut(values.loc[valid], 4, duplicates='drop')
    stats = subset.groupby('quartile', observed=True)['brier'].agg(['mean', 'std', 'count'])
    ci = 1.96 * stats['std'] / np.sqrt(stats['count'])
    x = np.arange(len(stats))
    xlabels = [f'Q{i+1}' for i in x]
    xlabels[0] += ' low'
    xlabels[-1] += ' high'
    ax.errorbar(x, stats['mean'], yerr=ci, fmt='o-', capsize=4, color='#c06b32')
    for xi, (_, row) in zip(x, stats.iterrows()):
        ax.annotate(f"n={row['count']:,}", (xi, row['mean']), xytext=(0, 7), textcoords='offset points', ha='center', fontsize=8)
    ax.set(xticks=x, xticklabels=xlabels, xlabel='Exposure quartile', ylabel='Mean Brier score', title=title)
    ax.grid(axis='y', alpha=.25)
fig.savefig(ROOT / 'figures/brier_by_market_metrics.png', dpi=180)
plt.show()

In [ ]:
# Calibration by within-sport lifetime-volume tertile for the seven largest sports.
# This is a separate cell so it can be rerun independently after the classification cell above.
VOLUME_CALIBRATION_DIR = ROOT / 'figures/calibration_by_sport_volume'
VOLUME_CALIBRATION_DIR.mkdir(parents=True, exist_ok=True)
top_7_sports = data.loc[data['sport'].ne('Other / unclassified')].groupby('sport').size().nlargest(7).index
volume_tier_order = ['Low volume', 'Mid volume', 'High volume']
volume_tier_summary = []

for sport in top_7_sports:
    sport_data = data.loc[data['sport'].eq(sport)].copy()
    sport_data['volumeNum'] = pd.to_numeric(sport_data['volumeNum'], errors='coerce')
    sport_data = sport_data.loc[sport_data['volumeNum'].notna() & sport_data['volumeNum'].ge(0)].copy()
    # Rank-based tertiles give three similarly sized groups; tied volume values may straddle a boundary.
    sport_data['volume_tier'] = pd.qcut(sport_data['volumeNum'].rank(method='first'), q=3, labels=volume_tier_order)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4.8), sharex=True, sharey=True, constrained_layout=True)
    grid = np.linspace(0, 1, 101)
    for ax, tier in zip(axes, volume_tier_order):
        tier_data = sport_data.loc[sport_data['volume_tier'].eq(tier)].copy()
        tier_data['bin'] = np.minimum((tier_data['probability'].clip(0, 1) * 10).astype(int), 9)
        calibration_bins = tier_data.groupby('bin').agg(n=('actual', 'size'), mean_probability=('probability', 'mean'), observed_rate=('actual', 'mean'))
        calibration_bins = calibration_bins.loc[calibration_bins['n'].gt(0)].copy()
        bn, br, bz = calibration_bins['n'].to_numpy(), calibration_bins['observed_rate'].to_numpy(), 1.96
        bden = 1 + bz**2 / bn
        bcenter = (br + bz**2 / (2*bn)) / bden
        bmargin = bz * np.sqrt(br*(1-br)/bn + bz**2/(4*bn**2)) / bden
        ci_low, ci_high = bcenter - bmargin, bcenter + bmargin
        Xv = np.column_stack([np.ones(len(calibration_bins)), calibration_bins['mean_probability'].to_numpy()])
        wv = np.sqrt(calibration_bins['n'].to_numpy())
        slope_text = 'n/a'
        if len(calibration_bins) >= 2:
            volume_intercept, volume_slope = np.linalg.lstsq(Xv * wv[:, None], calibration_bins['observed_rate'].to_numpy() * wv, rcond=None)[0]
            slope_text = f'{volume_slope:.2f}'
            ax.plot(grid, volume_intercept + volume_slope * grid, color='#c06b32', label='Weighted regression')
        else:
            ax.plot([], [], color='#c06b32', label='Regression unavailable (<2 bins)')
        ax.plot([0, 1], [0, 1], '--', color='0.45', label='Perfect calibration')
        yerr = np.maximum(np.vstack([calibration_bins['observed_rate'].to_numpy()-ci_low, ci_high-calibration_bins['observed_rate'].to_numpy()]), 0)
        ax.errorbar(calibration_bins['mean_probability'], calibration_bins['observed_rate'], yerr=yerr, fmt='none', ecolor='#3569a8', elinewidth=1.3, capsize=4, label='Wilson 95% CI', zorder=3)
        ax.scatter(calibration_bins['mean_probability'], calibration_bins['observed_rate'], s=20, color='#3569a8', edgecolor='white', linewidth=.5, label='10% bin mean', zorder=4)
        for _, row in calibration_bins.iterrows():
            ax.annotate(f"n={int(row['n']):,}", (row['mean_probability'], row['observed_rate']), xytext=(3, 3), textcoords='offset points', fontsize=7)
        tier_min, tier_max = tier_data['volumeNum'].min(), tier_data['volumeNum'].max()
        ax.set(title=f'{tier} (n={len(tier_data):,}; slope={slope_text})\nvolume {tier_min:,.0f}–{tier_max:,.0f}', xlim=(0, 1), ylim=(0, 1), xlabel='Predicted probability')
        ax.grid(alpha=.2)
        volume_tier_summary.append({'sport': sport, 'volume_tier': tier, 'markets': len(tier_data), 'volume_min': tier_min, 'volume_median': tier_data['volumeNum'].median(), 'volume_max': tier_max})
    axes[0].set_ylabel('Observed event frequency')
    handles, labels = axes[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, 1.08), ncol=4, frameon=True)
    safe_sport = ''.join(c.lower() if c.isalnum() else '_' for c in sport).strip('_')
    fig.suptitle(f'{sport}: calibration by lifetime-volume tier', y=1.14)
    fig.savefig(VOLUME_CALIBRATION_DIR / f'{safe_sport}_volume_tertiles.png', dpi=180, bbox_inches='tight')
    plt.show()
    plt.close(fig)

print('Top 7 sports by classified market count:', ', '.join(top_7_sports))
display(pd.DataFrame(volume_tier_summary).set_index(['sport', 'volume_tier']).round(2))